# Phase 1 — Generate Augmented Training Data

**Pipeline**: FAISS few-shot retrieval → Qwen3.5-9B-Instruct-AWQ (vLLM) → character-span mapping

**What this does**: Takes 10,000 unannotated patient notes and pseudo-labels them using a
large teacher model. The output `augmented_train.csv` supplements the 14,300 gold-labelled
rows in `train.csv` for Phase 2 fine-tuning.

| | |
|---|---|
| **Hardware** | Single T4 16 GB GPU (Colab Free Tier) |
| **Runtime** | ~2–4 hours |
| **Output** | `augmented_train.csv` — same schema as `train.csv` |

In [ ]:
import subprocess, sys

pkgs = [
    "numpy",                      # pin first — many packages depend on it
    "pyarrow",                    # required by pd.to_parquet / pd.read_parquet
    "vllm",                        # updated: use latest (0.19.1+ for Qwen3.5 support)
    "transformers>=5.5.0",
    "sentence-transformers",
    "faiss-cpu",
    "rapidfuzz",
    "pydantic",
    "tqdm",
    "pandas",
    "xgrammar",
]

print("Installing packages (this takes ~2 min on first run) ...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade", "--quiet", *pkgs
])
print("✓ Installation complete")
print()
print("━" * 60)
print("  ⚠  Restart the runtime now:  Runtime → Restart session")
print("  Then re-run all cells from the top.")
print("━" * 60)

# Auto-restart (uncomment if you want the kernel to restart automatically)
# import IPython; IPython.Application.instance().kernel.do_shutdown(True)

Installing packages (this takes ~2 min on first run) ...


In [ ]:
# Hugging Face login — required to download Qwen3.5-9B-Instruct-AWQ
# Get your token at: https://huggingface.co/settings/tokens
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Verify competition data files are present
# Upload via: Colab file browser, or kaggle API (see README)
import os
for f in ['features.csv', 'patient_notes.csv', 'train.csv']:
    status = "✓ found" if os.path.exists(f) else "✗ MISSING — upload before running main()"
    print(f"  {f}: {status}")

  features.csv: ✓ found
  patient_notes.csv: ✓ found
  train.csv: ✓ found


## Configuration

Edit the keys below before running. Most defaults are fine for Colab T4.

| Key | Default | When to change |
|-----|---------|----------------|
| `SAMPLE_SIZE` | `10_000` | Lower to `1_000` for a quick smoke test |
| `GPU_MEM_UTIL` | `0.82` | Lower to `0.75` if you hit OOM during vLLM init |
| `MAX_MODEL_LEN` | `4096` | Lower to `2048` if OOM persists |
| `FUZZY_SCORE_CUTOFF` | `72` | Raise to reduce false-positive span matches |

In [ ]:
import ast, gc, json, logging, re, sys
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
from pydantic import BaseModel
from rapidfuzz import fuzz, process as rfprocess
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from vllm import LLM, SamplingParams
try:
    from vllm.sampling_params import GuidedDecodingParams
except ImportError:
    GuidedDecodingParams = None
    print("WARNING: GuidedDecodingParams not found — restart runtime after pip install")

CONFIG = {
    "DATA_DIR":              Path("."),
    "OUTPUT_FILE":           Path("augmented_train.csv"),
    "FAISS_INDEX_FILE":      Path("faiss_features.index"),
    "FAISS_META_FILE":       Path("faiss_metadata.parquet"),
    "SAMPLE_SIZE":           10_000,
    "RANDOM_SEED":           42,
    "EMBED_MODEL":           "all-MiniLM-L6-v2",
    "EMBED_BATCH_SIZE":      64,
    "TOP_K_EXAMPLES":        3,
    # Official Qwen3.5-9B model for vLLM 0.19.1+
    "LLM_MODEL":             "Qwen/Qwen3.5-9B",
    "LLM_QUANTIZATION":      None,  # Use fp16 (no quantization for stability)
    "LLM_DTYPE":             "float16",
    "GPU_MEM_UTIL":          0.25,  # Very conservative: ~35GB total (30GB free + some overhead)
    "MAX_MODEL_LEN":         4096,
    "MAX_NEW_TOKENS":        300,
    "LLM_TEMPERATURE":       0.0,
    "GENERATION_BATCH_SIZE": 256,
    "FUZZY_SCORE_CUTOFF":    72,
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)

print("✓ Imports and CONFIG loaded")

✓ Imports and CONFIG loaded


## Section 1 — Data Loading & Filtering

Loads `train.csv`, `features.csv`, and `patient_notes.csv`.

Filters out the ~14,300 notes that are already annotated in `train.csv` — we only
want to pseudo-label the **unannotated** portion of the 42,146-note corpus.
Then randomly samples `SAMPLE_SIZE` (default 10,000) of those unannotated notes.

In [ ]:
def load_and_filter_data(cfg: dict) -> tuple:
    """
    Returns
    -------
    sample_notes : pd.DataFrame  — unannotated notes sampled for labelling
    train_df     : pd.DataFrame  — train.csv with annotation/location as Python lists
    features_df  : pd.DataFrame  — features.csv
    pn_df        : pd.DataFrame  — full patient_notes.csv (needed for FAISS metadata)
    """
    log.info("Loading CSV files ...")
    data_dir    = cfg["DATA_DIR"]
    train_df    = pd.read_csv(data_dir / "train.csv")
    features_df = pd.read_csv(data_dir / "features.csv")
    pn_df       = pd.read_csv(data_dir / "patient_notes.csv")

    def safe_parse_list(val):
        if pd.isna(val):
            return []
        try:
            result = ast.literal_eval(str(val))
            return result if isinstance(result, list) else []
        except (ValueError, SyntaxError):
            return []

    train_df["annotation"] = train_df["annotation"].apply(safe_parse_list)
    train_df["location"]   = train_df["location"].apply(safe_parse_list)

    annotated_pn_nums = set(train_df["pn_num"].unique())
    log.info(f"  Annotated notes in train.csv : {len(annotated_pn_nums)}")

    unannotated = pn_df[
        ~pn_df["pn_num"].isin(annotated_pn_nums)
        & pn_df["pn_history"].notna()
        & (pn_df["pn_history"].str.strip() != "")
    ].copy()
    log.info(f"  Unannotated notes available  : {len(unannotated)}")

    n_sample     = min(cfg["SAMPLE_SIZE"], len(unannotated))
    sample_notes = unannotated.sample(n=n_sample, random_state=cfg["RANDOM_SEED"]).reset_index(drop=True)
    log.info(f"  Sampled for labelling        : {len(sample_notes)}")

    return sample_notes, train_df, features_df, pn_df

print("✓ Section 1: load_and_filter_data defined")

✓ Section 1: load_and_filter_data defined


## Section 2 — FAISS Vector Index (Few-Shot Retrieval)

Builds a FAISS `IndexFlatIP` (inner-product / cosine similarity) over all annotated
examples in `train.csv`. Each vector encodes `"Feature: <text>  Annotation: <text>"`.

At generation time, for each `(feature, note)` pair we retrieve the **3 most similar**
labelled examples to use as few-shot context in the LLM prompt.

> **Caching**: The index is saved to `faiss_features.index` + `faiss_metadata.parquet`
> on first build. Re-running the notebook skips the rebuild and loads from disk in seconds.

In [ ]:
def build_faiss_index(train_df, pn_df, features_df, cfg) -> tuple:
    idx_path  = cfg["FAISS_INDEX_FILE"]
    meta_path = cfg["FAISS_META_FILE"]

    if idx_path.exists() and meta_path.exists():
        log.info("Loading cached FAISS index ...")
        index    = faiss.read_index(str(idx_path))
        metadata = pd.read_parquet(meta_path).to_dict("records")
        log.info(f"  Loaded {index.ntotal} vectors (dim={index.d})")
        return index, metadata

    log.info("Building FAISS index from train.csv ...")
    pn_map   = pn_df.set_index("pn_num")["pn_history"].to_dict()
    feat_map = features_df.set_index(["case_num", "feature_num"])["feature_text"].to_dict()

    embed_texts, metadata = [], []
    for _, row in train_df.iterrows():
        feature_text   = feat_map.get((row["case_num"], row["feature_num"]), "")
        pn_history     = pn_map.get(row["pn_num"], "")
        annotation_str = " | ".join(a for a in row["annotation"] if isinstance(a, str) and a.strip())
        if not feature_text or not annotation_str:
            continue
        embed_texts.append(f"Feature: {feature_text}  Annotation: {annotation_str}")
        metadata.append({
            "feature_text": feature_text,
            "annotation":   annotation_str,
            "pn_history":   (pn_history or "")[:500],
            "location":     str(row["location"]),
        })

    log.info(f"  Embedding {len(embed_texts)} train examples ...")
    embed_model = SentenceTransformer(cfg["EMBED_MODEL"])
    embeddings  = embed_model.encode(
        embed_texts, batch_size=cfg["EMBED_BATCH_SIZE"],
        show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True,
    ).astype(np.float32)
    del embed_model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    log.info(f"  FAISS index: {index.ntotal} vectors, dim={dim}")
    faiss.write_index(index, str(idx_path))
    pd.DataFrame(metadata).to_parquet(meta_path, index=False)
    log.info("  Index cached to disk.")
    return index, metadata


def retrieve_few_shot_examples(query_feature_text, index, metadata, embed_model, top_k=3) -> list:
    query_vec = embed_model.encode(
        [f"Feature: {query_feature_text}  Annotation:"],
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype(np.float32)
    distances, indices = index.search(query_vec, top_k)
    return [metadata[i] for i in indices[0] if 0 <= i < len(metadata)]

print("✓ Section 2: build_faiss_index, retrieve_few_shot_examples defined")

✓ Section 2: build_faiss_index, retrieve_few_shot_examples defined


## Section 3 — Prompt Construction

Each prompt is a 3-message chat conversation:
1. **System**: instructs the model to extract verbatim spans as JSON
2. **User**: includes 3 FAISS-retrieved examples + the target note + the target feature
3. *(Assistant turn added at generation time by vLLM)*

The `/no_think` suffix disables Qwen3.5's chain-of-thought mode so the model
outputs clean JSON without wrapping `<think>...</think>` blocks.

In [ ]:
SYSTEM_PROMPT = (
    "You are a clinical NLP specialist. "
    "Given a patient note and a clinical feature, extract the EXACT verbatim text spans "
    "from the note that express that feature. "
    "Rules:\n"
    "  1. Copy text character-for-character — do NOT paraphrase.\n"
    "  2. If the feature is absent from the note, return an empty list.\n"
    "  3. Output ONLY valid JSON — no markdown, no explanation, no <think> blocks.\n"
    'Output format: {"spans": ["exact text 1", "exact text 2"]}'
)


def build_messages(feature_text, pn_history, few_shot_examples) -> list:
    examples_block = ""
    for i, ex in enumerate(few_shot_examples, start=1):
        note_excerpt = ex["pn_history"][:300].replace("\n", " ").strip()
        ann_parts    = [a.strip() for a in ex["annotation"].split(" | ") if a.strip()]
        ann_json     = json.dumps(ann_parts)  # valid JSON with double-quoted strings
        examples_block += (
            f"\n[Example {i}]\n"
            f"Note (excerpt): \"{note_excerpt}\"\n"
            f"Feature: {ex['feature_text']}\n"
            f'Answer: {{"spans": {ann_json}}}\n'
        )

    target_note  = pn_history.replace("\n", " ").strip()
    user_content = (
        f"Here are labelled examples:{examples_block}\n"
        f"---\n"
        f"Now label this note.\n"
        f"Note: \"{target_note}\"\n"
        f"Feature: {feature_text}\n\n"
        "/no_think"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]

print("✓ Section 3: SYSTEM_PROMPT, build_messages defined")

✓ Section 3: SYSTEM_PROMPT, build_messages defined


## Section 4 — Span → Character Position Mapping

The LLM outputs text strings like `"substernal pressure"`. The competition requires
character offsets like `"42 62"`. This section maps extracted strings back to their
exact position in the original patient note.

**Three-step strategy** (most accurate first):
1. Exact substring match
2. Case-insensitive exact match
3. `rapidfuzz` sliding window — handles minor spacing/casing differences

In [ ]:
def _strip_think_tokens(text: str) -> str:
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def find_span_locations(span_texts, pn_history, fuzzy_cutoff=72) -> list:
    if not span_texts or not pn_history:
        return []

    locations, pn_lower = [], pn_history.lower()

    for span in span_texts:
        span = span.strip()
        if not span:
            continue

        # 1. Exact match
        idx = pn_history.find(span)
        if idx != -1:
            locations.append(f"{idx} {idx + len(span)}")
            continue

        # 2. Case-insensitive
        idx = pn_lower.find(span.lower())
        if idx != -1:
            locations.append(f"{idx} {idx + len(span)}")
            continue

        # 3. Fuzzy sliding window
        span_len = len(span)
        min_win  = max(1, int(span_len * 0.80))
        max_win  = min(len(pn_history), int(span_len * 1.20))

        best_score, best_start, best_end = 0, -1, -1
        for win_size in range(min_win, max_win + 1):
            n_windows  = len(pn_history) - win_size + 1
            if n_windows <= 0:
                continue
            candidates = [pn_history[s: s + win_size] for s in range(n_windows)]
            result = rfprocess.extractOne(span, candidates, scorer=fuzz.ratio, score_cutoff=fuzzy_cutoff)
            if result is not None:
                _text, score, pos = result
                if score > best_score:
                    best_score, best_start, best_end = score, pos, pos + win_size

        if best_score >= fuzzy_cutoff and best_start != -1:
            locations.append(f"{best_start} {best_end}")

    return locations

print("✓ Section 4: _strip_think_tokens, find_span_locations defined")

✓ Section 4: _strip_think_tokens, find_span_locations defined


## Section 5 — LLM Initialisation

Initialises vLLM with Qwen3.5-9B-Instruct-AWQ. Key T4 memory settings:

- **AWQ 4-bit**: keeps model at ~6–8 GB VRAM on a 16 GB T4
- `gpu_memory_utilization=0.82`: reserves ~3 GB headroom for KV cache
- `dtype="float16"`: T4 lacks BF16 tensor cores
- **XGrammar** constrained decoding: guarantees `{"spans": [...]}` on every output — no post-hoc JSON repair needed

In [ ]:
class SpanOutput(BaseModel):
    spans: list[str]


def init_llm(cfg: dict):
    """Initialize using transformers (simpler, notebook-safe)."""
    if not torch.cuda.is_available():
        log.warning("CUDA not available — inference will be very slow on CPU.")
    
    log.info(f"Loading model={cfg['LLM_MODEL']} with transformers...")
    from transformers import AutoModelForCausalLM, AutoTokenizer
    
    tokenizer = AutoTokenizer.from_pretrained(cfg["LLM_MODEL"], trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        cfg["LLM_MODEL"],
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    
    # Move to GPU if available
    if torch.cuda.is_available():
        model = model.cuda()
    
    log.info("✓ Model loaded")
    return model, tokenizer


def make_sampling_params(cfg: dict) -> dict:
    """Sampling params for transformers.generate()."""
    return {
        "max_new_tokens": cfg["MAX_NEW_TOKENS"],
        "temperature": cfg["LLM_TEMPERATURE"],
        "do_sample": False,
    }

print("✓ Section 5: SpanOutput, init_llm, make_sampling_params defined")

## Section 6 — Main Generation Loop

Three phases:

**Phase A — Build prompts**: For each `(unannotated note × feature)` pair, retrieve 3 FAISS
examples and build the chat message list. This produces all `N` prompts up front.

**Phase B — Batch LLM inference**: Send all prompts through `llm.chat()` in batches of 256.
vLLM's continuous batching means larger batches = better GPU utilisation.

**Phase C — Parse + map spans**: Parse the JSON output from each prediction, then map the
extracted text strings to `"start end"` character offsets using Section 4's logic.

In [ ]:
def generate_pseudo_labels(
    sample_notes, train_df, features_df,
    faiss_index, faiss_metadata, model, tokenizer, sampling_params, cfg,
) -> pd.DataFrame:
    feat_map = features_df.set_index(["case_num", "feature_num"])["feature_text"].to_dict()
    case_features = features_df.groupby("case_num")["feature_num"].apply(list).to_dict()

    log.info("Loading sentence-transformer for FAISS retrieval ...")
    embed_model = SentenceTransformer(cfg["EMBED_MODEL"])

    # ── Phase A: Build prompts ────────────────────────────────────────────────
    log.info("Phase A — Building (note × feature) prompts ...")
    all_messages, all_meta = [], []

    for _, note_row in tqdm(sample_notes.iterrows(), total=len(sample_notes), desc="Building prompts"):
        pn_num, case_num = int(note_row["pn_num"]), int(note_row["case_num"])
        pn_history       = note_row["pn_history"]
        if not isinstance(pn_history, str) or not pn_history.strip():
            continue

        for feature_num in case_features.get(case_num, []):
            feature_text = feat_map.get((case_num, feature_num), "")
            if not feature_text:
                continue
            few_shot = retrieve_few_shot_examples(
                feature_text, faiss_index, faiss_metadata, embed_model, cfg["TOP_K_EXAMPLES"]
            )
            all_messages.append(build_messages(feature_text, pn_history, few_shot))
            all_meta.append({"pn_num": pn_num, "case_num": case_num,
                             "feature_num": feature_num, "feature_text": feature_text,
                             "pn_history": pn_history})

    log.info(f"  Total (note × feature) pairs: {len(all_messages)}")
    del embed_model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    # ── Phase B: Batch LLM inference ─────────────────────────────────────────
    log.info("Phase B — Running batch LLM inference ...")
    all_raw_outputs = []
    batch_size = cfg["GENERATION_BATCH_SIZE"]
    n_batches  = (len(all_messages) + batch_size - 1) // batch_size

    for b_idx in tqdm(range(n_batches), desc="LLM batches"):
        start, end = b_idx * batch_size, min((b_idx + 1) * batch_size, len(all_messages))
        try:
            # Generate for batch
            batch_messages = all_messages[start:end]
            for msgs in batch_messages:
                # Format as chat conversation
                text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
                inputs = tokenizer(text, return_tensors="pt").to(model.device)
                
                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=sampling_params["max_new_tokens"],
                        temperature=sampling_params["temperature"],
                        do_sample=sampling_params["do_sample"],
                    )
                
                # Decode output
                response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
                all_raw_outputs.append(response)
        except Exception as exc:
            log.error(f"Batch {b_idx} failed: {exc}")
            all_raw_outputs.extend([""] * (end - start))
        
        if (b_idx + 1) % 10 == 0:
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

    # ── Phase C: Parse JSON + map spans ──────────────────────────────────────
    log.info("Phase C — Parsing outputs and mapping spans to character offsets ...")
    aug_rows = []

    for output, meta in tqdm(zip(all_raw_outputs, all_meta), total=len(all_raw_outputs), desc="Span mapping"):
        pn_history  = meta["pn_history"]
        pn_num      = meta["pn_num"]
        feature_num = meta["feature_num"]
        case_num    = meta["case_num"]
        row_id      = f"{pn_num:05d}_{feature_num:03d}"

        spans = []
        if output:
            try:
                raw_text = _strip_think_tokens(output.strip())
                # Robust JSON parsing with fallback
                try:
                    spans = [s for s in json.loads(raw_text).get("spans", [])
                            if isinstance(s, str) and s.strip()]
                except json.JSONDecodeError:
                    log.debug(f"JSON parse failed, raw: {raw_text[:100]}")
                    spans = []
            except (AttributeError, IndexError, TypeError) as e:
                log.debug(f"Output parse error: {e}")
                spans = []

        locations      = find_span_locations(spans, pn_history, cfg["FUZZY_SCORE_CUTOFF"])
        annotation_col = spans     if spans     else [""]
        location_col   = locations if locations else [""]

        aug_rows.append({
            "id": row_id, "pn_num": pn_num, "feature_num": feature_num,
            "case_num": case_num,
            "annotation": str(annotation_col), "location": str(location_col),
        })

    aug_df     = pd.DataFrame(aug_rows)
    non_empty  = (aug_df["location"] != str([""])).sum()
    fill_rate  = 100.0 * non_empty / max(len(aug_df), 1)
    log.info(f"Generated {len(aug_df)} rows | non-empty labels: {non_empty} ({fill_rate:.1f}%)")
    return aug_df

print("✓ Section 6: generate_pseudo_labels defined")

✓ Section 6: generate_pseudo_labels defined


In [ ]:
## Pre-cleanup before Phase 1

print("▶ Pre-cleanup: clearing GPU memory...")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
print("✓ GPU memory cleared")

▶ Pre-cleanup: clearing GPU memory...


✓ GPU memory cleared


In [ ]:
def main():
    cfg = CONFIG
    model = None
    tokenizer = None
    try:
        print("\n" + "="*65)
        print("  PHASE 1: Pseudo-Label Generation")
        print("="*65 + "\n")

        # Step 1: Load data
        print("▶ Step 1/4 — Loading and filtering data ...")
        sample_notes, train_df, features_df, pn_df = load_and_filter_data(cfg)
        print(f"  ✓ Sample notes: {len(sample_notes)} | Train: {len(train_df)} | Features: {len(features_df)}")

        # Step 2: FAISS index
        print("\n▶ Step 2/4 — Building FAISS index ...")
        faiss_index, faiss_metadata = build_faiss_index(train_df, pn_df, features_df, cfg)
        print(f"  ✓ Index: {faiss_index.ntotal} vectors")

        # Step 3: Init model
        print("\n▶ Step 3/4 — Loading model (downloading first time, ~5-10 min) ...")
        model, tokenizer  = init_llm(cfg)
        sampling_params   = make_sampling_params(cfg)
        print("  ✓ Model loaded")

        # Step 4: Generate
        print("\n▶ Step 4/4 — Generating pseudo-labels (~2-4 hours) ...")
        aug_df = generate_pseudo_labels(
            sample_notes, train_df, features_df,
            faiss_index, faiss_metadata, model, tokenizer, sampling_params, cfg,
        )

        # Save
        out_path = cfg["OUTPUT_FILE"]
        aug_df.to_csv(out_path, index=False)
        print(f"\n✓ Saved → {out_path}  shape={aug_df.shape}")

        # Summary only (suppress large output)
        non_empty = (aug_df["location"] != str([""])).sum()
        fill_rate = 100.0 * non_empty / max(len(aug_df), 1)
        print(f"  Non-empty labels: {non_empty} ({fill_rate:.1f}%)")
        print(f"  First 3 rows:\n{aug_df.head(3).to_string()}")

        print("\n" + "="*65)
        print("  ✓ Phase 1 complete — augmented_train.csv ready for Phase 2")
        print("="*65)

    except Exception as e:
        print(f"\n✗ Error during execution: {e}")
        raise
    finally:
        # Clean up resources
        print("\n▶ Cleaning up resources ...")
        if model is not None:
            del model
        if tokenizer is not None:
            del tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        print("  ✓ Resources cleaned up")

main()


  PHASE 1: Pseudo-Label Generation

▶ Step 1/4 — Loading and filtering data ...
2026-04-25 20:35:43,339  [INFO]  Loading CSV files ...
2026-04-25 20:35:43,794  [INFO]    Annotated notes in train.csv : 1000
2026-04-25 20:35:43,821  [INFO]    Unannotated notes available  : 41146
2026-04-25 20:35:43,829  [INFO]    Sampled for labelling        : 10000
  ✓ Sample notes: 10000 | Train: 14300 | Features: 143

▶ Step 2/4 — Building FAISS index ...
2026-04-25 20:35:43,829  [INFO]  Loading cached FAISS index ...
2026-04-25 20:35:43,881  [INFO]    Loaded 9901 vectors (dim=384)
  ✓ Index: 9901 vectors

▶ Step 3/4 — Loading model (downloading first time, ~5-10 min) ...
2026-04-25 20:35:43,881  [INFO]  Loading model=Qwen/Qwen3.5-9B with transformers...

✗ Error during execution: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`

▶ Cleaning up resources ...
  ✓ Resource

ValueError: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`